# Laboratorio 1 · Bitácora

**Nombre:**  
**Usuario de GitHub:**  
**Fecha:**  

---

> Los enunciados están en la guía del laboratorio. Aquí solo van tus
> predicciones, tus resultados y tus explicaciones.

> **La regla que hace que esto sirva de algo:** la predicción se escribe
> *antes* de ejecutar. Si la rellenas después ya sabiendo el resultado, el
> ejercicio no mide nada y tú no aprendes nada. Nadie va a comprobarlo:
> es un trato contigo mismo.


In [9]:
from rlrs.dp import greedy_policy, q_value, value_iteration
from rlrs.envs import ACTION_NAMES, GridWorld
from rlrs.evaluation import compare, evaluate
from rlrs.policies import GreedyTabularPolicy, RandomPolicy

# Este cuaderno es una bitácora: no define algoritmos, los usa.
# Si necesitas escribir una función que valga la pena conservar,
# va en src/rlrs/, no aquí.


## Mi variante

Ejecuta `uv run python scripts/variante.py` y anota lo que te tocó.

> Obtenido de ejecutar script variante que entrega los valores para el laboratorio
```
  Variante de  charliemedinar
    ruido           0.2
    coste por paso  -0.1
    gamma           0.9   (igual para todos)
```



In [ ]:
RUIDO  = 0.2   # <- rellena
COSTE  = -0.1   # <- rellena
GAMMA  = 0.9

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)
mi_env.noise, mi_env.step_reward


(0.2, -0.1)

---
## Ejercicio 1 · El respaldo a mano


**Antes de ejecutar.** ¿Cuál de las cuatro acciones crees que gana en (0,3), y por qué?

> Representacion de la grilla
```
 . . . 🤖 🏁
 . 🧱 . . 💀
 . . . 🧱 .
 . . . . .
```

🤖 Agente.
🧱 Obstaculo.
🏁 Meta
💀 Trampa

_Tu predicción:_  La acción que gana es ir a la derecha porque es la que representa "La Meta" para el agente en su entorno, es decir gana con ese movimiento


In [ ]:
valores, politica, barridos = value_iteration(mi_env, gamma=0.9)

for a in range(mi_env.n_actions):
    print(f'{ACTION_NAMES[a]:<10} {q_value(mi_env, valores, (0, 3), a, 0.9):+.6f}')


mi_env 4
ACTION_NAMES ('arriba', 'derecha', 'abajo', 'izquierda')
valores [ 0.25749673  0.45131822  0.65289019  0.90700188  0.          0.10414347
  0.          0.45654137  0.50413007  0.         -0.01834655  0.09245929
  0.26047376  0.         -0.27892568 -0.12814579 -0.03781283  0.07944004
 -0.05219899 -0.17877647]
politica [1 1 1 1 0 0 0 0 0 0 0 1 0 0 2 0 0 0 3 3]
barridos 35
arriba     +0.721801
derecha    +0.907002
abajo      +0.431734
izquierda  +0.497083


**Explica.** ¿Coincidió? ¿Por qué gana esa y no las otras?

_Tu respuesta:_ La opción de ir a la derecha desde la posición (0,3) representa ganar, tiene la mejor probabilidad frente a las demas porque estas te alejan del objetivo


---
## Ejercicio 2 · Tu variante, medida


In [16]:
print(f'{barridos} barridos · V(3,0) = {valores[mi_env.state_index((3, 0))]:+.4f}')
print(mi_env.render_values(valores, politica))

for r in compare(mi_env,
                 [GreedyTabularPolicy(politica, name='optima'),
                  RandomPolicy(mi_env.n_actions, seed=0)],
                 episodes=300, base_seed=0):
    print(' ', r)

35 barridos · V(3,0) = -0.1281
+0.26>  +0.45>  +0.65>  +0.91>   +1    
+0.10^    ###   +0.46^  +0.50^   -1    
-0.02^  +0.09>  +0.26^    ###   -0.28v 
-0.13^  -0.04^  +0.08^  -0.05<  -0.18< 
  optima         retorno +0.151 [+0.109, +0.193]  exito 98.0%  pasos   9.1
  aleatoria      retorno -5.881 [-6.232, -5.530]  exito 25.7%  pasos  56.8


**Anota.** Barridos, V(3,0), retorno con su intervalo, tasa de éxito y pasos medios.

_Tus números:_

- Barridos hasta converger: **35**
- V(3,0) = **-0.1281**
- Retorno medio (política óptima): **+0.151**, IC 95% **[+0.109, +0.193]**
- Tasa de éxito: **98.0%**
- Pasos medios: **9.1**

(referencia — política aleatoria: retorno -5.881 [-6.232, -5.530], éxito 25.7%, pasos 56.8)


---
## Ejercicio 3 · Subir gamma


**Antes de ejecutar.** Al pasar de 0,9 a 0,99: ¿qué le pasa al número de barridos? ¿Y a la política?

_Tu predicción:_ Con gamma más alto (0.99) el algoritmo va a necesitar más barridos, porque el criterio de parada depende de que el cambio entre un barrido y el siguiente baje de la tolerancia, y con menos descuento cada barrido reduce menos ese cambio. La política probablemente cambie poco o nada, porque mi costo por paso (-0.1) ya es bastante fuerte y el camino más corto ya gana claramente incluso con gamma=0.9.


In [ ]:
# Ejercicio 3: repetimos value_iteration sobre TU MISMO entorno (mi_env),
# cambiando únicamente gamma, de 0.9 a 0.99, para aislar el efecto del
# descuento y nada más (ruido y coste por paso quedan igual que antes).
GAMMA_ALTO = 0.99

valores_99, politica_99, barridos_99 = value_iteration(mi_env, gamma=GAMMA_ALTO)

print(f'gamma=0.90 -> {barridos} barridos')
print(f'gamma={GAMMA_ALTO} -> {barridos_99} barridos')

# Comparamos la política casilla por casilla: ¿en cuántos estados
# (que no sean muro ni terminal) cambió la acción óptima al subir gamma?
distintos = [
    i for i in range(mi_env.n_states)
    if not mi_env.is_wall(mi_env.state_pos(i))
    and not mi_env.is_terminal(mi_env.state_pos(i))
    and politica[i] != politica_99[i]
]
print(f'\nCasillas donde cambió la acción óptima: {len(distintos)} de {mi_env.n_states}')
for i in distintos:
    print(' ', mi_env.state_pos(i), ACTION_NAMES[politica[i]], '->', ACTION_NAMES[politica_99[i]])

print('\nRejilla con gamma = 0.90 (la de antes):')
print(mi_env.render_values(valores, politica))

print('\nRejilla con gamma =', GAMMA_ALTO, ':')
print(mi_env.render_values(valores_99, politica_99))


**Explica.** ¿Qué se movió mucho y qué se movió poco? ¿Por qué?

_Tu respuesta:_ Con mi variante, gamma=0.9 convergió en 35 barridos y gamma=0.99 en 40: subió, como esperaba. Esto pasa porque el operador de Bellman es una contracción de factor gamma — cuanto más cerca de 1, más lento se achica el error en cada barrido, así que hacen falta más rondas para bajar de la tolerancia.

La política, en cambio, no cambió en ninguna casilla (0 de 20 estados). Tiene sentido: mi costo por paso (-0.1) es bastante más duro que el de la guía (-0.04), así que ya con gamma=0.9 el agente prefiere fuertemente los caminos cortos. Subir gamma hace que el agente valore un poco más el futuro, pero no lo suficiente como para cambiar ninguna decisión: el camino óptimo ya era el mismo de antes. Lo que sí se movió fue el número puro de barridos (velocidad de convergencia), no la decisión final (la política).


---
## Ejercicio 4 · Quitar el ruido


**Antes de ejecutar.** Con ruido 0, ¿cambia la política óptima respecto a la tuya? ¿En qué casillas?

_Tu predicción:_ Con ruido 0.0, respecto a la política de ruido 0.2 (el entorno de la guía), creo que cambia en **0 casillas**. Bajar el ruido no cambia la geometría del tablero ni dónde está la meta o la trampa: solo hace que los valores sean más altos (porque hay menos chance de resbalar hacia algo malo), pero la dirección que ya era mejor bajo ruido debería seguir siendo la mejor sin ruido.


In [ ]:
# Ejercicio 4: entorno DE LA GUÍA (GridWorld() por defecto, no mi_env),
# variando solo el ruido, para ver cómo cambia la política óptima con
# lo predecible que es el mundo.
resultados = {}
for ruido in (0.0, 0.2, 0.6):
    e = GridWorld(noise=ruido)
    v, p, n = value_iteration(e, gamma=0.9)
    resultados[ruido] = (e, v, p, n)
    print(f'\nruido = {ruido}   {n} barridos')
    print(e.render_values(v, p))

# Comparamos la política de ruido 0.0 contra la de ruido 0.2, casilla por
# casilla, para contar exactamente cuántas cambiaron (no "algunas").
e0, _, p0, _ = resultados[0.0]
e2, _, p2, _ = resultados[0.2]
distintos_02 = [
    i for i in range(e0.n_states)
    if not e0.is_wall(e0.state_pos(i))
    and not e0.is_terminal(e0.state_pos(i))
    and p0[i] != p2[i]
]
print(f'\nCasillas donde cambia la política (ruido 0.0 vs 0.2): {len(distintos_02)}')
for i in distintos_02:
    print(' ', e0.state_pos(i))

# Extra: lo mismo pero contra ruido 0.6, para ver si con mucho ruido
# sí empieza a valer la pena rodear los peligros por otro lado.
e6, _, p6, _ = resultados[0.6]
distintos_62 = [
    i for i in range(e6.n_states)
    if not e6.is_wall(e6.state_pos(i))
    and not e6.is_terminal(e6.state_pos(i))
    and p6[i] != p2[i]
]
print(f'\nCasillas donde cambia la política (ruido 0.6 vs 0.2): {len(distintos_62)}')
for i in distintos_62:
    print(' ', e6.state_pos(i))


**Explica.** ¿Acertaste? Si te sorprendió, di exactamente qué esperabas y qué pasó.

_Tu respuesta:_ Acerté: entre ruido 0.0 y ruido 0.2 la política no cambió en ninguna casilla (0 de 16). Los barridos sí cambiaron bastante — 9 con ruido 0.0, 35 con ruido 0.2 — porque más ruido significa más incertidumbre sobre a dónde termino cayendo, y el "rumor" de value iteration tarda más en asentarse cuando cada movimiento tiene más resultados posibles que promediar.

Lo que sí me sorprendió fue subir el ruido hasta 0.6: ahí la política SÍ cambió, en 5 casillas, y tardó 90 barridos en converger. Con tanto ruido, moverse hacia una dirección arriesgada (cerca de la trampa) deja de valer la pena aunque sea el camino más corto, porque la chance de resbalar hacia el peligro es demasiado alta — el agente prefiere rodear por un camino más largo pero más seguro. Es la misma idea del piso con hielo: entre más resbaladizo está, más lejos te conviene mantenerte del borde.

---
## Ejercicio 5 · Encarecer el paso


**Antes de ejecutar.** Con coste por paso −2, ¿qué hará el agente?

_Tu predicción:_ Espero una tasa de éxito **menor** que con −0,04, quizás bastante menor. Con un castigo tan fuerte por paso, cada movimiento extra pesa muchísimo en la cuenta total; sospecho que al agente le puede convenir terminar el episodio lo antes posible, aunque eso signifique caer en la trampa (−1 una sola vez), en vez de aguantar varios pasos carísimos (−2 cada uno) para llegar hasta la meta.


In [ ]:
for coste in (-0.001, -0.04, -2.0):
    e = GridWorld(step_reward=coste)
    v, p, n = value_iteration(e, gamma=0.9)
    ev = evaluate(GridWorld(step_reward=coste), GreedyTabularPolicy(p), episodes=300, base_seed=0)
    print(f'coste {coste:>7} · {n:>2} barridos · {ev}')
    print(e.render_values(v, p), '\n')


**Explica.** ¿Qué está optimizando exactamente el agente para comportarse así?

_Tu respuesta:_ Acerté la dirección, pero no la magnitud: la tasa de éxito no solo bajó, se desplomó — de 98,0% (coste −0,04) a apenas **6,0%** (coste −2,0). Con coste −0,001 casi no hay castigo por moverse, y el éxito es prácticamente perfecto (100,0%).

Mirando la rejilla con coste −2,0, se ve por qué: en la fila de arriba, justo al lado de la trampa, la flecha en (1,3) apunta HACIA la trampa (`>`), no hacia la meta. El agente no está optimizando "llegar a la meta": está optimizando la **suma total de recompensa**, y esas dos cosas dejan de coincidir cuando seguir vivo cuesta demasiado. Pagar −2 varias veces más para alcanzar el +1 de la meta sale más caro que terminar ya con el −1 de la trampa, así que la política "óptima" elige morir rápido en vez de vivir caro. Es la misma lección de `divergencia.py`: el problema no es el algoritmo, es qué se está pagando — por el proceso (cada paso) en vez de por el resultado (llegar bien).


---
## Ejercicio 6 · El error plantado


In [ ]:
# ejecuta experiments/divergencia.py desde la terminal y pega aquí lo que salga



  UPTC · Sesion 1 · Que sostiene la convergencia

  1) value_iteration(env, gamma=1.0)
     ValueError: gamma debe estar en [0, 1); se recibio 1.0
     La guardia protege una garantia: sin gamma < 1 el operador de
     Bellman deja de ser una contraccion.

  2) gamma = 1.0, recompensa por paso -0.04  (el entorno de siempre)
   barrido       V(3,0)       max|V|       cambio
  ------------------------------------------------
         1      -0.0400       0.7920     0.792000
         5      -0.2000       0.9513     0.337498
        10       0.4938       0.9685     0.158584
        50       0.6675       0.9721     0.000000
       100       0.6675       0.9721     0.000000
       500       0.6675       0.9721     0.000000
      1000       0.6675       0.9721     0.000000
      2000       0.6675       0.9721     0.000000
     Converge. Quedarse dando vueltas cuesta -0.04 por paso, o sea
     -infinito, asi que ninguna politica que el max prefiera lo hace.
     Es un camino mas corto estocastico, y ahi gamma = 1 esta bien
     definido. Perder la garantia no es perder la convergencia.

  3) gamma = 1.0, recompensa por paso +0.01  (ahora le pagamos por moverse)
   barrido       V(3,0)       max|V|       cambio
  ------------------------------------------------
         1       0.0100       0.8020     0.802000
         5       0.0500       0.9703     0.368426
        10       0.8791       1.0174     0.180948
        50       1.4173       1.4173     0.010000
       100       1.9173       1.9173     0.010000
       500       5.9173       5.9173     0.010000
      1000      10.9173      10.9173     0.010000
      2000      20.9173      20.9173     0.010000
     No converge. A partir del barrido 100 los valores crecen +0.01
     por barrido, indefinidamente, y el cambio se estanca en 0.01:
     el criterio de parada nunca se dispara.

  El mismo entorno con gamma = 0.9 converge en 42 barridos,
  con max|V| = 0.9496. La unica diferencia es el descuento.

  El diagnostico tiene dos capas.
    Matematica: con gamma = 1 existe una politica que nunca termina y
    acumula +0.01 sin fin, luego su retorno es +infinito. No hay punto
    fijo finito al que converger.
    De diseno: el error no fue poner gamma = 1, fue pagar por el
    proceso en vez de por el resultado. El descuento solo lo tapaba.



**Explica las dos capas del diagnóstico.**

_La capa matemática:_ Con gamma = 1 no hay descuento, así que si existe alguna política que nunca termine el episodio y siga acumulando recompensa positiva paso a paso (aquí, +0.01 por moverse sin parar), su retorno total es infinito. La ecuación de Bellman busca un punto fijo finito al que converger, pero si el "mejor" valor posible es infinito, no hay punto fijo al que llegar — por eso los valores crecen sin parar (+0.01 por barrido, para siempre) y el criterio de parada (que espera que el cambio baje de una tolerancia) nunca se dispara. Es matemáticamente la pérdida de la garantía de contracción del operador de Bellman, que es lo que la guardia de `value_iteration` protege al exigir gamma < 1.

_La capa de diseño:_ El problema real no fue elegir gamma = 1 (eso, con el entorno de siempre y su costo -0.04 por paso, sí converge sin problema, en 42 barridos con gamma=0.9 y también con gamma=1 según el intento 2). El verdadero error fue **pagarle al agente por el proceso en vez de por el resultado**: al poner recompensa +0.01 por cada paso, se le está diciendo al agente "gana puntos por seguir moviéndote", y la respuesta óptima a esa señal es literalmente no parar nunca. El descuento (gamma < 1) tapaba el síntoma porque hacía que ese premio infinito valiera, en la práctica, un número finito — pero el diseño de la recompensa seguía mal. La lección es que hay que revisar primero qué se está incentivando con la función de recompensa, no solo ajustar gamma para que "converja".


---
## Cierre

**Lo que más me sorprendió hoy:** 

**Lo que todavía no entiendo:** 
